#### Bronze: ingest road status page
Downloads the NPS Blue Ridge Parkway road status page and stores it **unchanged**, as the raw source for the silver and gold layers.

**Outputs**
- Raw HTML file: `/Volumes/brp/bronze/raw/road_status_page/<run_date>/page.html`
- One record per day: `brp.bronze.road_status_raw`

**Steps**
1. **Settings:** load libraries and define the source URL, storage locations, and today's run date
2. **Download:** fetch the page; record the download time (UTC) and a SHA-256 fingerprint
3. **Save file:** write the untouched HTML to today's folder in the volume
4. **MERGE:** insert today's record into the bronze table, or replace it if it already exists
5. **Checks:** review stored snapshots and confirm there are no duplicate days

**Safe to re-run:** running twice on the same day updates that day's record instead of duplicating it.

##### Settings

In [0]:
# --- Configuration for the bronze ingest step ---
# Standard library
import os
import hashlib
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

# Third-party
import requests
from pyspark.sql import Row

# URL of the source web page (raw HTML will be stored as-is)
SOURCE_URL = "https://www.nps.gov/blri/planyourvisit/roadclosures.htm"

# Volume path where the raw HTML file is written
VOLUME_PATH = "/Volumes/brp/bronze/raw"

# Bronze Delta table that tracks one record per ingest run
BRONZE_TABLE = "brp.bronze.road_status_raw"

# Run date in the park's local time zone, used for file names and table records
run_date = datetime.now(ZoneInfo("America/New_York")).date().isoformat()
print("Run date:", run_date)

##### Download

In [0]:
# --- Download the source page unchanged ---
# Fetch the raw HTML from the NPS road status page, raising on any HTTP error.
response = requests.get(
    SOURCE_URL,
    timeout=30,
    headers={"User-Agent": "brp-closures-portfolio-project"},
)
response.raise_for_status()

# Capture the UTC fetch timestamp, the raw response bytes, and a SHA-256 fingerprint 
# of those bytes so this ingest run is auditable and reproducible 
# (the fingerprint lets later steps detect identical downloads).
fetched_at = datetime.now(timezone.utc)
raw_bytes = response.content
sha256 = hashlib.sha256(raw_bytes).hexdigest()

print("Fetch timestamp:", fetched_at)
print("Status:", response.status_code)
print("Size:", len(raw_bytes), "bytes")
print("Fingerprint:", sha256[:16], "...")

##### Save File

In [0]:
# --- Save the raw HTML file to the volume ---
# Write the unchanged response bytes to a date-partitioned folder under the bronze volume,
# preserving the exact content fetched from the source for auditability.
folder = f"{VOLUME_PATH}/road_status_page/{run_date}"
os.makedirs(folder, exist_ok=True)

file_path = f"{folder}/page.html"
with open(file_path, "wb") as f:
    f.write(raw_bytes)

print("Saved to:", file_path)

##### Merge

In [0]:

# --- Merge today's snapshot into the bronze table ---
# Build one record describing this run, then upsert it into the bronze table
# keyed by (source, run_date) so one row exists per source per day.
record = Row(
    source="road_status_page",
    url=SOURCE_URL,
    run_date=datetime.fromisoformat(run_date).date(),
    fetched_at=fetched_at,
    http_status=response.status_code,
    sha256=sha256,
    file_path=file_path,
    raw_html=response.text,
)

df = spark.createDataFrame([record])
# Register it as a temp view so the MERGE statement below can reference it.
df.createOrReplaceTempView("incoming")

# One row per source per day: update today's row if it exists, otherwise insert it.
spark.sql(f"""
    MERGE INTO {BRONZE_TABLE} AS t
    USING incoming AS s
    ON  t.source   = s.source
    AND t.run_date = s.run_date
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")
print("Merged today's snapshot into", BRONZE_TABLE)

##### Checks

In [0]:
%sql
-- Data-quality check: show what is stored in the bronze table, newest day first.
-- Used to confirm today's download landed and looks reasonable.
SELECT
  run_date,                              -- pipeline day (America/New_York)
  fetched_at,                            -- exact download time (UTC)
  http_status,                           -- server reply code; 200 means OK
  sha256,                                -- fingerprint of the raw page
  length(raw_html) AS html_length        -- size of the page in characters; 0 or tiny = something went wrong
FROM brp.bronze.road_status_raw
ORDER BY run_date DESC                   -- most recent day at the top

In [0]:

%sql
-- Data-quality check: each source should have at most ONE row per day.
-- Groups rows by source and day, counts them, and shows only groups with more than one.
-- Expected result: NO ROWS. Any row returned here is a duplicate to investigate.
SELECT
  source,
  run_date,
  COUNT(*) AS rows_for_day               -- how many rows exist for this source on this day
FROM brp.bronze.road_status_raw
GROUP BY source, run_date                -- one group per source per day
HAVING COUNT(*) > 1                      -- keep only groups that have duplicates